# Model 2: YOLOv8s-seg (Instance Segmentation)

YOLOv8 small variant for instance segmentation, pre-trained on COCO.

**Key improvements over baseline 0.3 mAP:**
- Letterbox resize (preserves aspect ratio) instead of forced square
- 50 epochs with early stopping (not 3)
- Full augmentation suite: mosaic, mixup, copy-paste, HSV, flip, rotation
- Cosine LR schedule with warmup
- Instance segmentation (polygon masks) instead of detection-only
- Label smoothing for noisy TACO labels

In [1]:
import torch
import os

# Verify GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# =========================
# PATHS
# =========================

BASE_DIR = r"C:\Users\User\Desktop\Ipynb"
DATASET_DIR = os.path.join(BASE_DIR, "archive", "dataset_v2")
DATASET_YAML = os.path.join(DATASET_DIR, "dataset.yaml")

print(f"\nDataset YAML: {DATASET_YAML}")
print(f"Exists: {os.path.exists(DATASET_YAML)}")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM: 8.0 GB

Dataset YAML: C:\Users\User\Desktop\Ipynb\archive\dataset_v2\dataset.yaml
Exists: True


## Quick Data Sanity Check\n\nVerify dataset structure before training.

In [2]:
# =========================
# VERIFY DATASET STRUCTURE
# =========================

for split in ["train", "val", "test"]:
    img_dir = os.path.join(DATASET_DIR, "images", split)
    lbl_dir = os.path.join(DATASET_DIR, "labels", split)
    
    img_files = set(os.path.splitext(f)[0] for f in os.listdir(img_dir) if not f.startswith('.'))
    lbl_files = set(os.path.splitext(f)[0] for f in os.listdir(lbl_dir) if not f.startswith('.'))
    
    matched = img_files & lbl_files
    img_only = img_files - lbl_files
    lbl_only = lbl_files - img_files
    
    print(f"{split}: {len(img_files)} images, {len(lbl_files)} labels, "
          f"{len(matched)} matched, {len(img_only)} img-only, {len(lbl_only)} lbl-only")

# Check a sample label
sample_lbl = os.path.join(DATASET_DIR, "labels", "train", os.listdir(os.path.join(DATASET_DIR, "labels", "train"))[0])
with open(sample_lbl) as f:
    lines = f.readlines()
print(f"\nSample label ({os.path.basename(sample_lbl)}):")
print(f"  Objects: {len(lines)}")
if lines:
    parts = lines[0].strip().split()
    print(f"  First object: class={parts[0]}, polygon_coords={len(parts)-1} values")

train: 1050 images, 1050 labels, 1050 matched, 0 img-only, 0 lbl-only
val: 300 images, 300 labels, 300 matched, 0 img-only, 0 lbl-only
test: 150 images, 150 labels, 150 matched, 0 img-only, 0 lbl-only

Sample label (batch_10_000002.txt):
  Objects: 6
  First object: class=8, polygon_coords=12 values


## Train YOLOv8s-seg

Training with aggressive augmentation and proper hyperparameters.

| Parameter | Value | Why |
|-----------|-------|-----|
| imgsz | 1024 | Large objects in TACO, letterbox preserves aspect ratio |
| batch | 4 | Safe for 8GB VRAM at 1024px |
| workers | 2 | Limits concurrent full-res image decodes to prevent RAM exhaustion |
| epochs | 100 | Enough to converge with early stopping |
| mosaic | 1.0 | Creates synthetic crowded scenes from 4 images |
| copy_paste | 0.0 | Disabled: Ultralytics bug with empty-annotation images |
| patience | 50 | Stop if no improvement for 50 epochs |

## Convert COCO Annotations → YOLO Segmentation Format

The shared `labels/` folder contains bounding-box labels written by the RT-DETR notebook.
YOLOv8-seg needs polygon labels. This cell overwrites those files with proper segmentation coordinates from the COCO JSON `segmentation` field.


In [3]:
import json
from pathlib import Path
from collections import defaultdict

def coco_to_yolo_seg(ann_file, label_dir, split):
    os.makedirs(label_dir, exist_ok=True)
    with open(ann_file) as f:
        data = json.load(f)

    img_info = {img["id"]: img for img in data["images"]}
    anns_by_img = defaultdict(list)
    for ann in data["annotations"]:
        anns_by_img[ann["image_id"]].append(ann)

    skipped = 0
    for img_id, img in img_info.items():
        iw, ih = img["width"], img["height"]
        stem = Path(img["file_name"]).stem
        lines = []
        for ann in anns_by_img[img_id]:
            cat = ann["category_id"] - 1  # COCO 1-indexed → YOLO 0-indexed
            segs = ann.get("segmentation", [])
            if segs and isinstance(segs, list) and len(segs[0]) >= 6:
                # Use the first (largest) polygon
                seg = segs[0]
                coords = []
                for i in range(0, len(seg) - 1, 2):
                    coords.append(max(0.0, min(1.0, seg[i] / iw)))
                    coords.append(max(0.0, min(1.0, seg[i + 1] / ih)))
            else:
                # No polygon — fall back to bbox corners as a 4-point polygon
                x, y, bw, bh = ann["bbox"]
                if bw <= 0 or bh <= 0:
                    skipped += 1
                    continue
                coords = [
                    x / iw, y / ih,
                    (x + bw) / iw, y / ih,
                    (x + bw) / iw, (y + bh) / ih,
                    x / iw, (y + bh) / ih,
                ]
                coords = [max(0.0, min(1.0, v)) for v in coords]

            lines.append(f"{cat} " + " ".join(f"{v:.6f}" for v in coords))

        with open(os.path.join(label_dir, stem + ".txt"), "w") as f:
            f.write("\n".join(lines))

    print(f"  {split}: {len(img_info)} label files written, {skipped} annotations skipped")

print("Converting COCO → YOLO segmentation labels...")
for split in ["train", "val", "test"]:
    coco_to_yolo_seg(
        ann_file=os.path.join(DATASET_DIR, f"{split}_annotations.json"),
        label_dir=os.path.join(DATASET_DIR, "labels", split),
        split=split,
    )

# Verify the new format
sample_lbl = os.path.join(DATASET_DIR, "labels", "train",
                          os.listdir(os.path.join(DATASET_DIR, "labels", "train"))[0])
with open(sample_lbl) as f:
    first = f.readline().strip().split()
print(f"\nVerification — {os.path.basename(sample_lbl)}: {len(first)-1} coords per first object "
      f"({len(first)//2} polygon points)")
print("Done.")


Converting COCO → YOLO segmentation labels...
  train: 1050 label files written, 0 annotations skipped
  val: 300 label files written, 0 annotations skipped
  test: 150 label files written, 0 annotations skipped

Verification — batch_10_000002.txt: 12 coords per first object (6 polygon points)
Done.


In [4]:
import shutil
from ultralytics import YOLO

# Load YOLOv8s-seg pretrained on COCO
model = YOLO("yolov8s-seg.pt")

# Train with optimized settings
results = model.train(
    data=DATASET_YAML,
    epochs=100,
    imgsz=1024,
    batch=4,
    workers=2,   # limit concurrent full-res TACO image decodes to prevent RAM OOM
    device=0,

    # ---- Augmentation ----
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.0,   # disabled: Ultralytics bug with empty-annotation images
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.1,
    erasing=0.1,

    # ---- Optimizer ----
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=5,
    warmup_momentum=0.5,

    # ---- Scheduler & Regularization ----
    cos_lr=True,

    # ---- Loss weights ----
    box=7.5,
    cls=0.5,
    dfl=1.5,

    # ---- Segmentation specific ----
    # overlap_mask=False: overlap_mask=True crashes when mosaic produces a
    # batch tile with 0 surviving annotations (empty cls_tensor index error)
    overlap_mask=False,
    mask_ratio=4,

    # ---- Early stopping ----
    patience=50,

    # ---- Logging ----
    project=os.path.join(BASE_DIR, "runs", "yolov8_seg"),
    name="taco_v3",
    save=True,
    save_period=25,
    plots=True,
    verbose=True,
)

# Rename best.pt → best_yolov8.pt to avoid confusion with other models
weights_dir = os.path.join(BASE_DIR, "runs", "yolov8_seg", "taco_v3", "weights")
src = os.path.join(weights_dir, "best.pt")
dst = os.path.join(weights_dir, "best_yolov8.pt")
shutil.copy2(src, dst)
print(f"Saved best weights to: {dst}")
print("Training complete!")

Ultralytics 8.4.38  Python-3.12.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\User\Desktop\Ipynb\archive\dataset_v2\dataset.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.1, exist_ok=False, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolov8s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=taco_v36, nbs=64, nms=False, opset=None, opt

KeyboardInterrupt: 

## Validate on Test Set

In [ ]:
# =========================
# EVALUATE ON TEST SET
# =========================

# Load best model
best_model_path = os.path.join(BASE_DIR, "runs", "yolov8_seg", "taco_v3", "weights", "best_yolov8.pt")
best_model = YOLO(best_model_path)

# Run validation on test split
test_metrics = best_model.val(
    data=DATASET_YAML,
    split="test",
    imgsz=1024,
    batch=4,
    device=0,
    plots=True,
    save_json=True,
    verbose=True,
)

print("\n" + "=" * 50)
print("YOLOV8s-SEG TEST RESULTS")
print("=" * 50)
print(f"Box mAP50:        {test_metrics.box.map50:.4f}")
print(f"Box mAP50-95:     {test_metrics.box.map:.4f}")
print(f"Mask mAP50:       {test_metrics.seg.map50:.4f}")
print(f"Mask mAP50-95:    {test_metrics.seg.map:.4f}")
print("=" * 50)

# Per-class results
print("\nPer-class Box mAP50:")
for i, name in enumerate(test_metrics.names.values()):
    if i < len(test_metrics.box.maps):
        print(f"  {name}: {test_metrics.box.maps[i]:.4f}")

## Counting Evaluation & Inference Speed

In [ ]:
import json
import time
import numpy as np
from collections import defaultdict

# =========================
# COUNTING EVALUATION
# =========================

# Load test ground truth
with open(os.path.join(DATASET_DIR, "test_annotations.json")) as f:
    test_data = json.load(f)

gt_counts = defaultdict(int)
for ann in test_data["annotations"]:
    gt_counts[ann["image_id"]] += 1

img_id_by_name = {img["file_name"]: img["id"] for img in test_data["images"]}

# Run inference on test images
test_image_dir = os.path.join(DATASET_DIR, "images", "test")
test_files = sorted([f for f in os.listdir(test_image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

count_errors = []
inference_times = []
per_image_results = []

for fname in test_files:
    img_path = os.path.join(test_image_dir, fname)
    img_id = img_id_by_name.get(fname, -1)
    gt_count = gt_counts.get(img_id, 0)

    start = time.time()
    preds = best_model(img_path, imgsz=1024, device=0, verbose=False)
    elapsed = (time.time() - start) * 1000
    inference_times.append(elapsed)

    pred_count = len(preds[0].boxes) if preds[0].boxes is not None else 0
    error = abs(pred_count - gt_count)
    count_errors.append(error)

    per_image_results.append({
        "file": fname,
        "gt_count": gt_count,
        "pred_count": pred_count,
        "count_error": error,
        "inference_ms": elapsed,
    })

within_1 = sum(1 for e in count_errors if e <= 1) / len(count_errors) * 100
within_3 = sum(1 for e in count_errors if e <= 3) / len(count_errors) * 100

print("=" * 50)
print("YOLOV8s-SEG - COUNTING & SPEED")
print("=" * 50)
print(f"Counting MAE:          {np.mean(count_errors):.2f}")
print(f"Counting within +/-1:  {within_1:.1f}%")
print(f"Counting within +/-3:  {within_3:.1f}%")
print(f"Avg inference time:    {np.mean(inference_times):.1f} ms")
print(f"Median inference time: {np.median(inference_times):.1f} ms")
print("=" * 50)

# Save results
output_dir = os.path.join(BASE_DIR, "runs", "yolov8_seg", "taco_v3")
results_data = {
    "metrics": {
        "box_map50": float(test_metrics.box.map50),
        "box_map50_95": float(test_metrics.box.map),
        "mask_map50": float(test_metrics.seg.map50),
        "mask_map50_95": float(test_metrics.seg.map),
        "counting_mae": float(np.mean(count_errors)),
        "counting_within_1": float(within_1),
        "counting_within_3": float(within_3),
        "avg_inference_ms": float(np.mean(inference_times)),
    },
    "per_image": per_image_results,
}

with open(os.path.join(output_dir, "yolov8_seg_results.json"), "w") as f:
    json.dump(results_data, f, indent=2)
print(f"Results saved to: {output_dir}/yolov8_seg_results.json")

## Visualize Predictions

In [ ]:
import matplotlib.pyplot as plt
import cv2

# =========================
# VISUALIZE PREDICTIONS
# =========================

sample_files = test_files[:6] if len(test_files) >= 6 else test_files

fig, axes = plt.subplots(2, 3, figsize=(20, 14))
axes = axes.flatten()

for idx, fname in enumerate(sample_files):
    img_path = os.path.join(test_image_dir, fname)

    preds = best_model(img_path, imgsz=1024, device=0, verbose=False)
    result = preds[0]

    annotated = result.plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    img_id = img_id_by_name.get(fname, -1)
    gt_count = gt_counts.get(img_id, 0)
    pred_count = len(result.boxes) if result.boxes is not None else 0

    axes[idx].imshow(annotated_rgb)
    axes[idx].set_title(f"GT: {gt_count} | Pred: {pred_count}", fontsize=12)
    axes[idx].axis("off")

plt.suptitle("YOLOv8s-seg Predictions on Test Set (taco_v3)", fontsize=16)
plt.tight_layout()

output_dir = os.path.join(BASE_DIR, "runs", "yolov8_seg", "taco_v3")
viz_path = os.path.join(output_dir, "yolov8_seg_predictions.png")
plt.savefig(viz_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to: {viz_path}")

## Algorithm 4: Watershed Post-Processing

YOLOv8 provides bounding boxes and class labels. We discard its built-in low-resolution masks and replace them with precise watershed masks.

```
Trained YOLOv8-seg → boxes + class labels  (masks discarded)
  → For each box: crop image (+ padding) → watershed_on_crop()
  → Place mask back on full-image canvas
  → Evaluate: box mAP + mask mAP + binary IoU + counting
```

In [ ]:
# ── Watershed helpers (same as Algorithm 3) ───────────────────────────────────
import cv2
import numpy as np
import time
from collections import defaultdict
from tqdm import tqdm
from pycocotools import mask as coco_mask_util
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


def watershed_on_crop(crop, padding=10):
    h, w     = crop.shape[:2]
    gray     = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    filtered = cv2.bilateralFilter(crop, 9, 75, 75)
    a_chan   = cv2.cvtColor(filtered, cv2.COLOR_BGR2LAB)[:, :, 1]

    _, thresh_otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    thresh_a = cv2.adaptiveThreshold(a_chan, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY, 15, 2)
    edges    = cv2.Canny(gray, 30, 100)
    combined = cv2.bitwise_or(thresh_otsu, thresh_a)
    combined = cv2.add(combined, cv2.dilate(edges, np.ones((2, 2), np.uint8)))
    _, combined = cv2.threshold(combined, 127, 255, cv2.THRESH_BINARY)

    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    cleaned = cv2.morphologyEx(combined, cv2.MORPH_OPEN,  kernel, iterations=1)
    cleaned = cv2.morphologyEx(cleaned,  cv2.MORPH_CLOSE, kernel, iterations=2)

    bw = max(3, min(padding // 2, 8))
    border_mask = np.ones((h, w), dtype=np.uint8) * 255
    border_mask[:bw, :] = border_mask[-bw:, :] = 0
    border_mask[:, :bw] = border_mask[:, -bw:] = 0
    sure_bg = cv2.bitwise_or(cv2.dilate(cleaned, kernel, iterations=3), border_mask)

    dist = cv2.distanceTransform(cleaned, cv2.DIST_L2, 5)
    if dist.max() > 0:
        _, sure_fg = cv2.threshold(dist, 0.4 * dist.max(), 255, 0)
        sure_fg = np.uint8(sure_fg)
    else:
        sure_fg = np.zeros((h, w), dtype=np.uint8)
        cx, cy  = w // 2, h // 2
        r = min(cx, cy, w - cx, h - cy) // 2
        if r > 0:
            cv2.circle(sure_fg, (cx, cy), r, 255, -1)

    unknown  = cv2.subtract(sure_bg, sure_fg)
    _, marks = cv2.connectedComponents(sure_fg)
    marks    = marks + 1
    marks[unknown == 255] = 0
    marks    = cv2.watershed(crop, marks)

    mask  = np.zeros((h, w), dtype=np.uint8)
    valid = set(np.unique(marks)) - {-1, 1}
    if valid:
        best = max(valid, key=lambda lbl: np.sum(marks == lbl))
        mask[marks == best] = 255
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    else:
        mask = cleaned
    return mask


def apply_watershed_to_boxes(image, boxes, labels, scores, conf=0.25, padding=15):
    h, w = image.shape[:2]
    dets = []
    for box, label, score in zip(boxes, labels, scores):
        if score < conf:
            continue
        x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[2]), int(box[3])
        x1c = max(0, x1 - padding); y1c = max(0, y1 - padding)
        x2c = min(w, x2 + padding); y2c = min(h, y2 + padding)
        if x2c - x1c < 5 or y2c - y1c < 5:
            continue
        crop_mask = watershed_on_crop(image[y1c:y2c, x1c:x2c], padding=padding)
        full_mask = np.zeros((h, w), dtype=np.uint8)
        full_mask[y1c:y2c, x1c:x2c] = crop_mask
        dets.append({"box": [x1, y1, x2, y2], "class_id": int(label) + 1,  # YOLO 0-idx -> COCO 1-idx
                     "score": float(score), "mask": full_mask})
    return dets


def build_gt_mask_ws(anns, img_shape):
    combined = np.zeros(img_shape[:2], dtype=np.uint8)
    for ann in anns:
        for seg in ann.get("segmentation", []):
            pts = np.array(seg, dtype=np.float32).reshape(-1, 2).astype(np.int32)
            cv2.fillPoly(combined, [pts], 255)
    return combined


def evaluate_watershed_detections(det_by_image, coco_data, img_id_lookup):
    gt_counts = defaultdict(int)
    gt_masks  = defaultdict(list)
    for ann in coco_data["annotations"]:
        gt_counts[ann["image_id"]] += 1
        gt_masks[ann["image_id"]].append(ann)

    bbox_preds, segm_preds = [], []
    binary_ious, count_errs, per_image = [], [], []

    for image_id, (img_shape, fname, dets) in det_by_image.items():
        info   = img_id_lookup.get(image_id, {})
        orig_h = info.get("height", img_shape[0])
        orig_w = info.get("width",  img_shape[1])

        pred_m = np.zeros(img_shape[:2], dtype=np.uint8)
        for d in dets:
            pred_m = cv2.bitwise_or(pred_m, d["mask"])
        gt_m   = build_gt_mask_ws(gt_masks.get(image_id, []), img_shape)
        inter  = cv2.countNonZero(cv2.bitwise_and(pred_m, gt_m))
        union  = cv2.countNonZero(cv2.bitwise_or( pred_m, gt_m))
        binary_ious.append(inter / max(union, 1))

        gt_count = gt_counts.get(image_id, 0)
        count_errs.append(abs(len(dets) - gt_count))
        per_image.append({"file": fname, "gt_count": gt_count,
                          "pred_count": len(dets), "count_error": abs(len(dets) - gt_count)})

        for d in dets:
            x1, y1, x2, y2 = d["box"]
            bbox_preds.append({"image_id": image_id, "category_id": d["class_id"],
                               "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)],
                               "score": d["score"]})
            m = d["mask"].copy()
            if m.shape != (orig_h, orig_w):
                m = cv2.resize(m, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
            rle = coco_mask_util.encode(np.asfortranarray(m))
            rle["counts"] = rle["counts"].decode("utf-8")
            segm_preds.append({"image_id": image_id, "category_id": d["class_id"],
                               "segmentation": rle, "score": d["score"]})

    gt_coco = COCO(); gt_coco.dataset = coco_data; gt_coco.createIndex()
    b50 = b95 = m50 = m95 = 0.0
    if bbox_preds:
        dt = gt_coco.loadRes(bbox_preds)
        ev = COCOeval(gt_coco, dt, "bbox"); ev.evaluate(); ev.accumulate(); ev.summarize()
        b50, b95 = float(ev.stats[1]), float(ev.stats[0])
    if segm_preds:
        dt = gt_coco.loadRes(segm_preds)
        ev = COCOeval(gt_coco, dt, "segm"); ev.evaluate(); ev.accumulate(); ev.summarize()
        m50, m95 = float(ev.stats[1]), float(ev.stats[0])

    n = len(count_errs)
    metrics = {
        "box_map50":         b50,  "box_map50_95":       b95,
        "mask_map50":        m50,  "mask_map50_95":      m95,
        "binary_iou":        float(np.mean(binary_ious)),
        "counting_mae":      float(np.mean(count_errs)),
        "counting_within_1": float(sum(1 for e in count_errs if e <= 1) / n * 100),
        "counting_within_3": float(sum(1 for e in count_errs if e <= 3) / n * 100),
    }
    return metrics, per_image


print("Watershed helpers loaded.")

In [ ]:
CONF_WS    = 0.25
PADDING_WS = 15

# Load test annotations
with open(os.path.join(DATASET_DIR, "test_annotations.json")) as f:
    ws_test_coco = json.load(f)

ws_img_file_lookup = {img["file_name"]: img for img in ws_test_coco["images"]}
ws_img_id_lookup   = {img["id"]: img     for img in ws_test_coco["images"]}
ws_test_image_dir  = os.path.join(DATASET_DIR, "images", "test")
ws_test_files      = sorted([
    f for f in os.listdir(ws_test_image_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

# Load best weights (best_model may already be defined; re-load to be safe)
best_weights_path = os.path.join(BASE_DIR, "runs", "yolov8_seg", "taco_v3", "weights", "best_yolov8.pt")
yolo_ws_model = YOLO(best_weights_path)
print(f"Loaded: {best_weights_path}")

det_by_image = {}
det_times    = []

for fname in tqdm(ws_test_files, desc="YOLO+Watershed"):
    img_bgr  = cv2.imread(os.path.join(ws_test_image_dir, fname))
    if img_bgr is None:
        continue
    img_info = ws_img_file_lookup.get(fname)
    if img_info is None:
        continue
    image_id = img_info["id"]

    t0  = time.time()
    res = yolo_ws_model(img_bgr, imgsz=1024, conf=CONF_WS, verbose=False)[0]
    det_times.append((time.time() - t0) * 1000)

    if res.boxes and len(res.boxes) > 0:
        boxes  = res.boxes.xyxy.cpu().numpy()
        labels = res.boxes.cls.cpu().numpy().astype(int)
        scores = res.boxes.conf.cpu().numpy()
    else:
        boxes = np.zeros((0, 4)); labels = np.zeros(0, int); scores = np.zeros(0)

    dets = apply_watershed_to_boxes(img_bgr, boxes, labels, scores, CONF_WS, PADDING_WS)
    det_by_image[image_id] = (img_bgr.shape, fname, dets)

print("\nEvaluating…")
metrics, per_image = evaluate_watershed_detections(det_by_image, ws_test_coco, ws_img_id_lookup)
avg_inf = float(np.mean(det_times[1:]) if len(det_times) > 1 else det_times[0])

print(f"\n{'='*50}")
print("Algorithm 4: YOLOv8-seg + Watershed")
print(f"{'='*50}")
for k, v in metrics.items():
    print(f"  {k:25s}: {v:.4f}")
print(f"  {'avg_inference_ms':25s}: {avg_inf:.1f}")

ws_output_dir = os.path.join(BASE_DIR, "runs", "yolov8_seg")
results_data = {
    "metrics": {**metrics, "avg_inference_ms": avg_inf},
    "per_image": per_image,
}
out_path = os.path.join(ws_output_dir, "yolo_watershed_results.json")
with open(out_path, "w") as f:
    json.dump(results_data, f, indent=2)
print(f"\nSaved: {out_path}")